Imports & Config

In [3]:
import os
from pathlib import Path
from typing import List

from langchain_community.document_loaders import (
    PyPDFLoader, TextLoader, DirectoryLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma, FAISS
from langchain_ollama import ChatOllama          # or HuggingFacePipeline if you prefer
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

# Config
DATA_DIR = Path("./data")          # put your PDFs/Markdown/txt here
PERSIST_DIR = "./chroma_db"           # or "./faiss_index"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 5
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # fast & local
LLM_MODEL = "llama3.2"                # must be pulled via ollama

Cell 2 – Document Ingestion & Chunking

In [4]:
def load_documents(data_dir: Path) -> List[Document]:
    loaders = {
        ".pdf": PyPDFLoader,
        ".txt": TextLoader,
        ".md": TextLoader,
    }
    docs = []
    for ext, loader_cls in loaders.items():
        loader = DirectoryLoader(
            str(data_dir),
            glob=f"**/*{ext}",
            loader_cls=loader_cls,
            show_progress=True,
        )
        docs.extend(loader.load())
    return docs

raw_docs = load_documents(DATA_DIR)
print(f"Loaded {len(raw_docs)} documents")

# Chunk with overlap + keep source metadata
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    add_start_index=True,
)

chunks = text_splitter.split_documents(raw_docs)

# Enrich metadata
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
    # source is already present from the loader

print(f"Created {len(chunks)} chunks")

100%|██████████| 2/2 [00:07<00:00,  3.52s/it]
0it [00:00, ?it/s]
0it [00:00, ?it/s]

Loaded 24 documents
Created 34 chunks


Cell 3 – Embeddings + Vector Store (Chroma or FAISS)

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},          # change to "cuda" if you have GPU
    encode_kwargs={"normalize_embeddings": True},
)

# Option A: Chroma (persistent, easy)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
)
vectorstore.persist()

# Option B: FAISS (faster pure similarity, no server)
# vectorstore = FAISS.from_documents(chunks, embeddings)
# vectorstore.save_local("./faiss_index")

print("Index built and persisted")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Index built and persisted


C:\Users\divin\AppData\Local\Temp\ipykernel_17452\884830961.py:13: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


Retriever

In [6]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K},
)

# Quick sanity check
test_docs = retriever.invoke("What is the main topic of the documents?")
for d in test_docs:
    print(d.metadata.get("source"), "→", d.page_content[:120], "...")

data\Engi-Tech and PRD.pdf → Platform
 
Modules
 
&
 
Functional
 
Requirements
 
 
Industry-First
 
Consultation
 
Gateway
 
(
/contact
 
&
 
/indus ...
data\Engi-Tech and PRD.pdf → ●
 
Speed
 
Benchmark:
 
Core
 
Web
 
Vitals
 
target:
 
First
 
Contentful
 
Paint
 
(FCP)
 
le
 
1.0s,
 
Largest
 
Con ...
data\Engi-Tech and PRD.pdf → ○
 
Establish
 
brand
 
voice
 
adhering
 
strictly
 
to
 
problem-first,
 
industry-focused
 
positioning.
 
2.
 
Phase ...
data\Presentation - ENGI-TECH 2026 Goals.pdf → Q2(Apr – Jun): Training &
Deployments
Launch Cohort
Begin first student
training cohort this
quarter
Start Projects
Init ...
data\Engi-Tech and PRD.pdf → Core
 
Value
 
Proposition
 
(10-Second
 
Visitor
 
Clarity)
 
When
 
a
 
visitor
 
arrives
 
on
 
the
 
digital
 
platf ...


Prompt + Generation Chain

In [7]:
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0.1,          # low temp for grounded answers
    num_ctx=8192,             # adjust to your model / RAM
)

# Prompt that forces citations and grounded answers
prompt = ChatPromptTemplate.from_template(
"""You are a helpful assistant that answers questions using ONLY the provided context.
If the answer is not in the context, say "I don't have enough information in the knowledge base."
Always cite the source(s) using the format [source: filename] at the end of relevant sentences.
Also give a short confidence statement (High / Medium / Low).

Context:
{context}

Question: {question}

Answer:"""
)

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join(
        f"[source: {d.metadata.get('source', 'unknown')} | chunk {d.metadata.get('chunk_id')}]\n{d.page_content}"
        for d in docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

Query Interface

In [8]:
def ask(question: str):
    print(f"\nQ: {question}")
    answer = rag_chain.invoke(question)
    print(f"\nA: {answer}")
    return answer

# Examples
ask("Summarize the key points from the documents.")
ask("What specific facts or numbers are mentioned?")


Q: Summarize the key points from the documents.

A: I don't have enough information in the knowledge base.

Confidence: Low

Q: What specific facts or numbers are mentioned?

A: I can provide specific facts or numbers mentioned in the context.

Here are a few examples:

* The organization is targeting over 1,000 trained students (chunk 18).
* The organization aims to complete 10 to 15 client projects (chunk 30).
* The organization is developing 3 flagship internal products (chunk 30).
* The organization has a target for the Core Web Vitals (FCP) to be loaded in 1.0s and Largest Contentful Paint (LCP) in 2.0s (chunk 14).
* The organization has a target for SEO and indexing, targeting sector-specific search intents (chunk 14).
* The organization has a Phase 1 implementation roadmap for architecture and core messaging (chunk 1).

Confidence: High (all facts and numbers mentioned in the context are verifiable)


'I can provide specific facts or numbers mentioned in the context.\n\nHere are a few examples:\n\n* The organization is targeting over 1,000 trained students (chunk 18).\n* The organization aims to complete 10 to 15 client projects (chunk 30).\n* The organization is developing 3 flagship internal products (chunk 30).\n* The organization has a target for the Core Web Vitals (FCP) to be loaded in 1.0s and Largest Contentful Paint (LCP) in 2.0s (chunk 14).\n* The organization has a target for SEO and indexing, targeting sector-specific search intents (chunk 14).\n* The organization has a Phase 1 implementation roadmap for architecture and core messaging (chunk 1).\n\nConfidence: High (all facts and numbers mentioned in the context are verifiable)'